# 02.1 - Regularisation sweep for the joint reconstruction

02 ran two fixed settings, unregularised and one regularised point, and the regularised
result was not convincing. This notebook asks whether that is the method or the choice of
prior, by sweeping the regularisers instead of asserting one.

## What is swept, and why these three

| knob | what it penalises | prior it encodes |
|---|---|---|
| `tv_weight` | TV of each channel's own phase | the lattice is piecewise smooth |
| `tv_channel_diff` | $\mathrm{mean}\,\lvert\nabla(\varphi_{on}-\varphi_{off})\rvert$ | the **excitation** is piecewise constant |
| `channel_diff_l2_grad` | $\mathrm{mean}\,\lvert\nabla(\varphi_{on}-\varphi_{off})\rvert^{2}$ | the **excitation** is smooth |

The third is new. Isotropic TV is an $L_1$ on the gradient magnitude, which is
*edge-preserving*: it is cheap to keep a sharp step and expensive to keep a gentle ramp.
That is the right prior for a piecewise-constant image and the wrong one for the object we
actually have - the excitation is a smooth dipole with no edges in it, 4 Å across and
tapered. A squared-$L_2$ on the gradient inverts that preference: expensive for steps,
cheap for ramps. It was added to `ObjectMultiplexed` as
`get_multi_channel_l2_grad_loss`, next to the TV version it mirrors, and is reached
through `co_reconstruction_regularized(channel_diff_l2_grad=...)`.

## The fair fight

Comparing a *regularised* joint reconstruction against a *raw* reconstruct-and-subtract is
not a fair comparison - half the difference is just that one of them has been smoothed.
So 01's $\Delta\varphi$ is post-smoothed with a Gaussian over a range of widths and the
best of that family is what the joint results have to beat. Post-smoothing is weaker than
regularising inside the loop (it cannot use the data to decide what to keep), which is the
point: it is the honest floor, not a straw man.

## Scoring

The matched-filter metrics from 09, restated here so this notebook stands alone. Every
numerator is an inner product against the truth, never a peak - a peak is an extreme-value
statistic that rises with noise and will rank a noisier result higher.

* `scale` $= \langle r,t\rangle/\langle t,t\rangle$ - how much of the true amplitude came back
* `rho` - Pearson correlation, shape fidelity independent of scale
* `nrmse` $= \lVert r-t\rVert/\lVert t\rVert$
* `snr` $= \lvert a\rvert\lVert t\rVert/\mathrm{std}(r-at)$

**09 remains the canonical definition.** If they change there, change them here.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, map_coordinates

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"
RESULTS_DIR = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(NOTEBOOK_DIR.parent))
import method_wrappers
from quantem.core import config

config.set_device(0)

with open(DATA_DIR / "ideal_geometry.json") as handle:
    META = json.load(handle)
truth = np.load(DATA_DIR / "ideal_truth.npz")
delta_phi_true = truth["delta_phi_true"].astype(np.float64)

scan_params = META["scan"]["scan_params_for_method_wrappers"]
scan_step_ang = META["scan"]["step_ang"]
n_scan = META["scan"]["n_positions_per_side"]
object_dx_ang = META["cell"]["dx_ang"]
n_object = META["cell"]["n_pixels"]
object_x_ang = (np.arange(n_object) - n_object // 2) * object_dx_ang
blob_centre = np.array(META["object"]["blob_centre_ang"])

ii, jj = np.indices((n_scan, n_scan))
scan_mask = ((ii + jj) % 2 == 0).astype(np.uint8)

print(f"scan   : {n_scan}^2 at {scan_step_ang} A")
print(f"truth  : peak |delta phi| = {np.abs(delta_phi_true).max():.5f} rad")


def load_merged(dose_key):
    return np.load(DATA_DIR / f"ideal_merged_dose_{dose_key}.npy")


def centred_axis(n, pixel):
    return (np.arange(n) - n / 2 + 0.5) * pixel


def to_common_grid(array, pixel, axis_common):
    axis = centred_axis(array.shape[0], pixel)
    coords = (axis_common - axis[0]) / pixel
    gy, gx = np.meshgrid(coords, coords, indexing="ij")
    return map_coordinates(array.astype(np.float64),
                           np.vstack([gy.ravel(), gx.ravel()]),
                           order=1, mode="nearest").reshape(gy.shape)


def compare(recovered, truth_grid):
    """Matched-filter metrics; same definitions as 09."""
    r = recovered - recovered.mean()
    t = truth_grid - truth_grid.mean()
    tt = float(np.vdot(t, t).real)
    scale = float(np.vdot(t, r).real) / tt
    residual = r - scale * t
    denom = residual.std()
    return dict(scale=scale,
                rho=float(np.corrcoef(r.ravel(), t.ravel())[0, 1]),
                nrmse=float(np.linalg.norm(r - t) / np.linalg.norm(t)),
                snr=float(abs(scale) * np.linalg.norm(t) / denom) if denom > 0 else np.inf)

cuda:0
scan   : 200^2 at 0.15 A
truth  : peak |delta phi| = 0.02106 rad


## Parameters

In [2]:
DOSE = "inf"             # sweep at one dose; widen once the picture is clear

NUM_ITERS = 60           # calibrated in 01/02: obj_lr 1e-3 is flat by ~20 iterations
OBJ_LR = 1e-3
PROBE_LR = 1e-4
BATCH_SIZE = 128

TV_OBJ = 1.0             # the per-channel object prior, held fixed while the
                         # differential priors are swept

# one knob at a time, against a common baseline
SWEEP = (
    [dict(label="baseline (no differential prior)", tv_channel_diff=0.0,
          channel_diff_l2_grad=0.0)]
    + [dict(label=f"TV diff = {w:g}", tv_channel_diff=w, channel_diff_l2_grad=0.0)
       for w in (1.0, 10.0, 100.0)]
    + [dict(label=f"L2-grad diff = {w:g}", tv_channel_diff=0.0,
            channel_diff_l2_grad=w) for w in (1.0, 10.0, 100.0)]
)

# post-smoothing widths for 01's output, in Angstrom
SMOOTH_SIGMAS_ANG = [0.0, 0.3, 0.6, 1.0, 1.5, 2.5]

print(f"{len(SWEEP)} joint reconstructions at dose {DOSE}, {NUM_ITERS} iterations each")
for s in SWEEP:
    print(f"  {s['label']}")

7 joint reconstructions at dose inf, 60 iterations each
  baseline (no differential prior)
  TV diff = 1
  TV diff = 10
  TV diff = 100
  L2-grad diff = 1
  L2-grad diff = 10
  L2-grad diff = 100


## The sweep

In [3]:
merged = load_merged(DOSE)
runs = {}
for cfg in SWEEP:
    label = cfg["label"]
    print(f"\n=== {label} ===")
    t0 = time.time()
    res = method_wrappers.co_reconstruction_regularized(
        merged, scan_mask, scan_params,
        num_iters=NUM_ITERS, tv_weight=TV_OBJ,
        tv_channel_diff=cfg["tv_channel_diff"],
        channel_diff_l2_grad=cfg["channel_diff_l2_grad"],
        obj_lr=OBJ_LR, probe_lr=PROBE_LR, batch_size=BATCH_SIZE,
    )
    runs[label] = dict(delta_phi=res["delta_phi"],
                       pixel_size_ang=res["pixel_size_ang"],
                       **{k: cfg[k] for k in ("tv_channel_diff", "channel_diff_l2_grad")})
    print(f"  {time.time()-t0:.0f} s, peak |delta phi| = "
          f"{np.abs(res['delta_phi']).max():.5f} rad")
    del res
    torch.cuda.empty_cache()
del merged
print(f"\n{len(runs)} reconstructions done")


=== baseline (no differential prior) ===


Forcing best fit rotation to 0 degrees.
Forcing transpose of intensities to False.


Normalizing intensities:   0%|          | 0/40000 [00:00<?, ?probe position/s]

/home/rotem/ptycho_light/quantem/src/quantem/diffractive_imaging/object_models.py:1317: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/Copy.cpp:308.)
  post_constraint_obj[ch] = self.apply_hard_constraints(


  0%|          | 0/60 [00:00<?, ?it/s]

/home/rotem/ptycho_light/quantem/src/quantem/diffractive_imaging/object_models.py:420: UserWarning: calculating TV loss for phase, need to check phase wrapping. Easiest fix is scalar phase array.
  warn(


... (26 outputs left out)


=== L2-grad diff = 100 ===


Forcing best fit rotation to 0 degrees.
Forcing transpose of intensities to False.


Normalizing intensities:   0%|          | 0/40000 [00:00<?, ?probe position/s]

  0%|          | 0/60 [00:00<?, ?it/s]

## The fair fight: 01's output, post-smoothed

In [ ]:
# 01's output, post-smoothed, as the floor the joint results have to clear.
sub_path = None
for tag in ("01_step2x", "01_step1x"):
    p = RESULTS_DIR / f"{tag}_dose_{DOSE}.npz"
    if p.exists():
        sub_path = p
        break

smoothed = {}
if sub_path is None:
    print("01 has not been run at this geometry - no subtract baseline to smooth.\n"
          "Run 01 first; the joint sweep above stands on its own in the meantime.")
else:
    with np.load(sub_path) as h:
        sub_delta = h["delta_phi"].astype(np.float64)
        sub_px = float(np.atleast_1d(h["pixel_size_ang"])[0])
    print(f"baseline from {sub_path.name}: {sub_delta.shape} at {sub_px:.5f} A/px")
    for sigma_ang in SMOOTH_SIGMAS_ANG:
        sigma_px = sigma_ang / sub_px
        img = sub_delta if sigma_px == 0 else gaussian_filter(sub_delta, sigma_px)
        smoothed[f"01 subtract, sigma = {sigma_ang:g} A"] = dict(
            delta_phi=img, pixel_size_ang=sub_px, sigma_ang=sigma_ang)
    print(f"{len(smoothed)} smoothing widths prepared")

## Score everything on one grid

In [ ]:
everything = {**runs, **smoothed}
assert everything, "nothing to score"

half_fov = min(v["delta_phi"].shape[0] / 2 * v["pixel_size_ang"] for v in everything.values())
half_fov = min(half_fov, object_x_ang[-1]) * 0.98
AXIS = np.arange(-half_fov, half_fov + 1e-9, object_dx_ang)
TRUTH = to_common_grid(delta_phi_true, object_dx_ang, AXIS)
TRUTH -= TRUTH.mean()
print(f"common grid {len(AXIS)}^2 at {object_dx_ang:.5f} A/px, +/-{half_fov:.2f} A\n")

rows = []
for label, v in everything.items():
    g = to_common_grid(v["delta_phi"], v["pixel_size_ang"], AXIS)
    m = compare(g, TRUTH)
    m["label"] = label
    v["grid"] = g
    rows.append(m)

rows.sort(key=lambda r: -r["snr"])
print(f"{'':38s} {'scale':>8s} {'rho':>8s} {'nrmse':>8s} {'snr':>9s}")
print("-" * 76)
for r in rows:
    print(f"{r['label']:38s} {r['scale']:8.4f} {r['rho']:8.4f} "
          f"{r['nrmse']:8.4f} {r['snr']:9.3f}")
print("-" * 76)
print("sorted by snr; scale 1.0 = full amplitude, rho 1.0 = perfect shape")

with open(RESULTS_DIR / f"021_sweep_dose_{DOSE}.json", "w") as handle:
    json.dump({"dose": DOSE, "num_iters": NUM_ITERS, "tv_obj": TV_OBJ,
               "metric_definitions": "see 09; every numerator is an inner product",
               "rows": [{k: (round(v, 6) if isinstance(v, float) else v)
                         for k, v in r.items()} for r in rows]}, handle, indent=2)
print(f"\nwrote 021_sweep_dose_{DOSE}.json")

In [ ]:
best_joint = next(r for r in rows if r["label"] in runs)
best_sub = next((r for r in rows if r["label"] in smoothed), None)

panels = [("ground truth", TRUTH, None)]
panels += [(r["label"], everything[r["label"]]["grid"], r)
           for r in rows[:5]]

ncol = 3
nrow = int(np.ceil(len(panels) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.0 * ncol, 4.6 * nrow),
                         constrained_layout=True, squeeze=False)
ext = [AXIS[0], AXIS[-1]] * 2
tvmax = float(np.abs(TRUTH).max())
for ax, (label, img, m) in zip(axes.ravel(), panels):
    g = img - img.mean()
    lim = tvmax if m is None else float(np.abs(g).max())
    im = ax.imshow(g.T, extent=ext, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.plot(*blob_centre, "k+", ms=11, mew=1.5)
    title = label if m is None else (f"{label}\nscale {m['scale']:.3f}  "
                                     r"$\rho$ " + f"{m['rho']:.3f}  snr {m['snr']:.2f}")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(r"x [$\AA$]"); ax.set_ylabel(r"y [$\AA$]")
    fig.colorbar(im, ax=ax, shrink=0.8)
for ax in axes.ravel()[len(panels):]:
    ax.axis("off")
fig.suptitle(f"02.1 - best five by SNR, dose {DOSE} (truth on its own scale)", fontsize=13)
plt.show()

# how each prior trades amplitude against shape
fig, axes = plt.subplots(1, 2, figsize=(14.0, 4.6), constrained_layout=True)
for ax, field in zip(axes, ("rho", "snr")):
    for key, marker, name in ((("tv_channel_diff"), "o-", "TV on the difference"),
                              (("channel_diff_l2_grad"), "s-", "L2-grad on the difference")):
        ws, vals = [], []
        for label, v in runs.items():
            other = ("channel_diff_l2_grad" if key == "tv_channel_diff"
                     else "tv_channel_diff")
            if v[other] != 0:
                continue
            ws.append(v[key])
            vals.append(next(r[field] for r in rows if r["label"] == label))
        order = np.argsort(ws)
        ax.plot(np.array(ws)[order], np.array(vals)[order], marker, lw=2, ms=7, label=name)
    if smoothed:
        best = max((r[field] for r in rows if r["label"] in smoothed))
        ax.axhline(best, color="#d62728", ls="--", lw=1.6,
                   label="best post-smoothed 01")
    ax.set_xscale("symlog", linthresh=1.0)
    ax.set_xlabel("regularisation weight"); ax.set_ylabel(field)
    ax.set_title(f"{field} against weight, one knob at a time", fontsize=11)
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.show()

## Reading this

The question is not which row wins but **whether the differential prior helps at all**,
and if so which shape of prior. Three outcomes are worth distinguishing:

* **Both priors flat in weight** - the regularisation is not what is limiting 02, and the
  problem is elsewhere (the probe, the iteration count, the gauge).
* **TV and L2-grad both help, similarly** - any smoothing helps, and the honest comparison
  is against the post-smoothed 01 line, not against raw 01.
* **L2-grad beats TV** - the prior shape matters, and TV was mismatched to a smooth
  excitation. This is the outcome the new regulariser was added to test.

The dashed line is the best post-smoothed 01. A joint result that does not clear it is not
earning its complexity, whatever its own sweep looks like.